# Indian Author Book Recommender — End-to-End Pipeline

**SMAI Assignment 3 · T14.2**

Runs the full pipeline as a single notebook: fetch → embed → index → query.
Mirrors the standalone scripts (`data_fetcher.py`, `build_index.py`, `recommender.py`) for the GitHub repo deliverable.

Time budget on a laptop: ~10–20 min, mostly Open Library latency.

## 0 · Install (first run only)

In [ ]:
# !pip install -q -r requirements.txt

In [ ]:
import os
from pathlib import Path

# Jupyter sets CWD to the notebook's directory; we need the project root.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
print("Working directory:", Path.cwd())

## 1 · Fetch corpus from Open Library

Skip this cell if `data/books.csv` already exists.

In [ ]:
import os
if not os.path.exists('data/books.csv'):
    import data_fetcher
    data_fetcher.main(out_path='data/books.csv', enrich=True, max_books=1500)
else:
    print('data/books.csv exists — skipping fetch.')

In [ ]:
import pandas as pd
df = pd.read_csv('data/books.csv').fillna('')
print(f'Books: {len(df)}')
print(f'With descriptions: {(df.description.str.len() > 0).sum()}')
print(f'With cover URL: {(df.cover_url.str.len() > 0).sum()}')
df.head(3)

## 2 · Encode + build FAISS index

In [ ]:
import build_index
build_index.main(csv_path='data/books.csv', out_dir='data')

## 3 · Sanity-check with a few queries

In [ ]:
from recommender import BookRecommender
rec = BookRecommender('data')

for q in [
    'partition stories about families divided across the border',
    'magical realism in modern Indian fiction',
    'mythological retelling with a strong female protagonist',
    'rural village life in south India',
    'satirical novel about Indian bureaucracy',
]:
    print(f'\n=== {q!r} ===')
    for r in rec.search(q, k=5):
        print(f"  {r['score']:.3f}  {r['title']} — {r['authors'][:60]}")

## 4 · Inspect a justification

In [ ]:
q = 'partition stories'
results = rec.search(q, k=3)
for r in results:
    print(f"\n{r['title']}  ({r['score']:.3f})")
    print(' ', rec.justify(q, r))

## 5 · (Optional) LLM-based justification

Uncomment if you have an Anthropic API key set in `ANTHROPIC_API_KEY`.

In [ ]:
# from anthropic import Anthropic
# client = Anthropic()
# for r in results:
#     print(f"\n{r['title']}")
#     print('  template:', rec.justify(q, r))
#     print('  llm     :', rec.justify_llm(q, r, client=client))